# Rainfall Statistics Analysis Using IMD Gridded Rainfall Dataset (2017 & 2018)

## Objective
The objective of this project is to analyze the IMD High Spatial Resolution (0.25° × 0.25°) Daily Gridded Rainfall Dataset for the years 2017 and 2018 using Python.

The analysis includes:
- Reading binary (.grd) rainfall files
- Cleaning the dataset
- Computing overall statistics
- Computing daily statistics
- Computing monthly statistics
- Saving the results into CSV files

## Import Required Libraries

In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print(os.getcwd())
print(os.listdir())


c:\Users\madhu\OneDrive\Desktop\IMD-DATA-PROCESSING\notebook
['Rainfall_Statistics_Analysis.ipynb']


## Read IMD Binary (.grd) File

The IMD rainfall data is stored as a binary `.grd` file.

Each file contains:
- 135 × 129 spatial grid
- Daily rainfall values
- Missing values represented by -999

In [2]:
def read_grd(filename, days):

    rows = 129
    cols = 135

    data = np.fromfile(filename, dtype=np.float32)

    data = data.reshape(days, rows, cols)

    data[data == -999] = np.nan

    return data

# Analysis of Rainfall Data for 2017

In [12]:
data2017 = read_grd("../data/Rainfall_ind2017_rfp25.grd",365)

print(data2017.shape)

(365, 129, 135)


## Overall Statistics (2017)

This section computes:

- Mean
- Median
- Standard Deviation
- Variance
- Minimum
- Maximum
- Range
- Percentiles

In [13]:
def overall_statistics(data):

    rain=data.flatten()

    rain=rain[~np.isnan(rain)]

    stats={

        "Mean":round(np.mean(rain),2),

        "Median":round(np.median(rain),2),

        "Standard Deviation":round(np.std(rain),2),

        "Variance":round(np.var(rain),2),

        "Minimum":round(np.min(rain),2),

        "Maximum":round(np.max(rain),2),

        "Range":round(np.max(rain)-np.min(rain),2),

        "25th Percentile":round(np.percentile(rain,25),2),

        "50th Percentile":round(np.percentile(rain,50),2),

        "75th Percentile":round(np.percentile(rain,75),2),

        "90th Percentile":round(np.percentile(rain,90),2),

        "95th Percentile":round(np.percentile(rain,95),2)

    }

    return pd.DataFrame(stats.items(),
                        columns=["Statistic","Value"])

In [14]:
overall2017 = overall_statistics(data2017)

overall2017

,Statistic,Value
0,Mean,3.100000
1,Median,0.000000
2,Standard Deviation,10.820000
3,Variance,117.089996
4,Minimum,0.000000
5,Maximum,540.270020
6,Range,540.270020
7,25th Percentile,0.000000
8,50th Percentile,0.000000
9,75th Percentile,0.370000


## Daily Statistics (2017)

The following statistics are calculated for each day:

- Mean
- Median
- Standard Deviation
- Variance
- Minimum
- Maximum

In [15]:
def daily_statistics(data):

    return pd.DataFrame({

        "Day":range(1,data.shape[0]+1),

        "Mean":np.nanmean(data,axis=(1,2)),

        "Median":np.nanmedian(data,axis=(1,2)),

        "Std":np.nanstd(data,axis=(1,2)),

        "Variance":np.nanvar(data,axis=(1,2)),

        "Minimum":np.nanmin(data,axis=(1,2)),

        "Maximum":np.nanmax(data,axis=(1,2))

    })

In [16]:
daily2017 = daily_statistics(data2017)

daily2017.head()

,Day,Mean,Median,Std,Variance,Minimum,Maximum
0,1,0.005683,0.0,0.224090,0.050216,0.0,12.768674
1,2,0.103776,0.0,0.912445,0.832555,0.0,32.774227
2,3,0.163319,0.0,1.278259,1.633945,0.0,31.725138
3,4,0.678239,0.0,3.414197,11.656740,0.0,29.400002
4,5,0.463675,0.0,2.489673,6.198473,0.0,22.900000


## Monthly Statistics (2017)

Monthly rainfall statistics include:

- Mean
- Median
- Standard Deviation
- Variance
- Minimum
- Maximum
- Total Rainfall

In [18]:
def monthly_statistics(data, leap=False):

    if leap:
        month_days = [31,29,31,30,31,30,31,31,30,31,30,31]
    else:
        month_days = [31,28,31,30,31,30,31,31,30,31,30,31]

    month_names = [
        "January","February","March","April","May","June",
        "July","August","September","October","November","December"
    ]

    results = []
    start = 0

    for month, days in zip(month_names, month_days):

        end = start + days
        temp = data[start:end]

        results.append({
            "Month": month,
            "Mean": round(np.nanmean(temp),2),
            "Median": round(np.nanmedian(temp),2),
            "Std": round(np.nanstd(temp),2),
            "Variance": round(np.nanvar(temp),2),
            "Minimum": round(np.nanmin(temp),2),
            "Maximum": round(np.nanmax(temp),2),
            "Total Rainfall": round(np.nansum(temp),2)
        })

        start = end

    return pd.DataFrame(results)

In [19]:
monthly2017 = monthly_statistics(data2017, leap=False)

monthly2017

,Month,Mean,Median,Std,Variance,Minimum,Maximum,Total Rainfall
0,January,0.62,0.00,3.830000,14.690000,0.0,184.020004,9.493145e+04
1,February,0.43,0.00,3.360000,11.290000,0.0,201.690002,6.007620e+04
2,March,1.00,0.00,5.040000,25.440001,0.0,245.050003,1.535611e+05
3,April,1.71,0.00,7.960000,63.299999,0.0,238.240005,2.545900e+05
4,May,1.94,0.00,6.670000,44.430000,0.0,212.649994,2.991233e+05
5,June,5.70,0.12,13.990000,195.580002,0.0,540.270020,8.493215e+05
6,July,9.40,1.85,18.760000,351.929993,0.0,424.700012,1.445922e+06
7,August,7.48,0.99,17.379999,301.980011,0.0,435.109985,1.151092e+06
8,September,5.07,0.00,12.510000,156.389999,0.0,331.140015,7.554042e+05
9,October,2.65,0.00,9.720000,94.470001,0.0,336.619995,4.082190e+05


## Save Results (2017)

In [21]:
overall2017.to_csv("../outputs/2017/overall_2017.csv",index=False)

daily2017.to_csv("../outputs/2017/daily_2017.csv",index=False)

monthly2017.to_csv("../outputs/2017/monthly_2017.csv",index=False)

# Analysis of Rainfall Data for 2018

In [23]:
data2018 = read_grd("../data/Rainfall_ind2018_rfp25.grd",365)

print(data2018.shape)

(365, 129, 135)


## Overall Statistics (2018)

In [24]:
overall2018 = overall_statistics(data2018)

overall2018

,Statistic,Value
0,Mean,2.760000
1,Median,0.000000
2,Standard Deviation,10.110000
3,Variance,102.120003
4,Minimum,0.000000
5,Maximum,534.250000
6,Range,534.250000
7,25th Percentile,0.000000
8,50th Percentile,0.000000
9,75th Percentile,0.000000


## Daily Statistics (2018)

In [25]:
daily2018 = daily_statistics(data2018)

daily2018.head()

,Day,Mean,Median,Std,Variance,Minimum,Maximum
0,1,0.006322,0.0,0.142237,0.020231,0.0,6.702260
1,2,0.055044,0.0,0.514562,0.264774,0.0,10.737739
2,3,0.205750,0.0,1.189552,1.415033,0.0,19.746922
3,4,0.123429,0.0,1.016136,1.032532,0.0,16.369471
4,5,0.007578,0.0,0.068896,0.004747,0.0,1.722966


## Monthly Statistics (2018)

In [26]:
monthly2018 = monthly_statistics(data2018, leap=False)

monthly2018

,Month,Mean,Median,Std,Variance,Minimum,Maximum,Total Rainfall
0,January,0.09,0.00,1.040000,1.090000,0.0,68.449997,1.376596e+04
1,February,0.36,0.00,2.460000,6.040000,0.0,98.040001,5.029048e+04
2,March,0.61,0.00,3.610000,13.060000,0.0,186.520004,9.334430e+04
3,April,1.35,0.00,4.800000,23.070000,0.0,113.370003,2.014276e+05
4,May,2.16,0.00,7.610000,57.970001,0.0,277.679993,3.322811e+05
5,June,5.32,0.00,13.910000,193.429993,0.0,332.829987,7.928065e+05
6,July,8.85,1.37,18.049999,325.769989,0.0,534.250000,1.362631e+06
7,August,7.44,0.92,16.170000,261.510010,0.0,364.510010,1.145058e+06
8,September,4.40,0.00,12.430000,154.509995,0.0,371.059998,6.550848e+05
9,October,1.18,0.00,6.760000,45.720001,0.0,287.220001,1.823332e+05


## Save Results (2018)

In [27]:
overall2018.to_csv("../outputs/2018/overall_2018.csv",index=False)

daily2018.to_csv("../outputs/2018/daily_2018.csv",index=False)

monthly2018.to_csv("../outputs/2018/monthly_2018.csv",index=False)

# Conclusion

The rainfall data for the years 2017 and 2018 was successfully analyzed using Python. Binary IMD `.grd` files were read and processed to compute overall, daily, and monthly rainfall statistics. The results were exported as CSV files for further analysis and visualization.

This workflow demonstrates how large meteorological datasets can be efficiently processed using NumPy and Pandas.